# Proyecto final: verificación de la categoría de piezas de mobiliario con nubes de puntos

**Autor:** Sebastian Baquero Alvarez (30000122682), Universidad de San Buenaventura Cali<br>
**Curso:** Electiva en Ingeniería de Software – Profesor Carlos Mario Paredes

Este notebook reproduce todos los resultados del informe final:

1. Descarga ModelNet10 y convierte cada malla CAD en una nube de 1024 puntos.
2. Calcula un vector de 146 características geométricas por objeto.
3. Entrena un perceptrón multicapa (Keras) y tres modelos de comparación, cada uno con 5 semillas.
4. Evalúa con exactitud, exactitud media por clase, F1 macro, matriz de confusión
   y un análisis de verificación (ROC/AUC) que simula la estación de inspección.
5. **Prueba de robustez:** evalúa los modelos con las piezas de prueba giradas, con ruido y con una
   parte tapada, para simular lo que entregaría un escáner real.
6. Guarda todas las figuras y un archivo `resultados.json` en la carpeta `salidas/`.

**Cómo correrlo:** en la carpeta del proyecto, `pip install -r requirements.txt` y luego
"Run All" en VS Code (o Jupyter). No necesita GPU. La primera vez descarga ModelNet10 (unos 450 MB)
y calcula las características; después usa la caché guardada en `salidas/`.

In [ ]:
import os, io, json, time, zipfile, urllib.request, random
import numpy as np
from scipy import stats
from scipy.spatial import cKDTree
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_recall_fscore_support, confusion_matrix,
                             roc_auc_score, roc_curve)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

import pandas as pd

print("TensorFlow", tf.__version__)

## 1. Configuración general
Todos los hiperparámetros quedan en un solo diccionario para que el experimento sea fácil de repetir
y para que el artículo pueda citarlos tal cual.

In [ ]:
CONFIG = {
    "semilla_datos": 42,
    "n_puntos": 1024,            # puntos muestreados por objeto
    "k_vecinos": 16,             # vecindario para descriptores locales
    "n_pares_d2": 4096,          # pares aleatorios para la distribución D2
    "bins_eje": 16, "bins_d1": 16, "bins_d2": 32, "bins_local": 8,
    "frac_validacion": 0.15,     # se toma del conjunto de entrenamiento oficial
    "capas_ocultas": [256, 128, 64],
    "dropout": [0.3, 0.3, 0.2],
    "l2": 1e-4,
    "tasa_aprendizaje": 1e-3,
    "batch_size": 32,
    "max_epocas": 300,
    "paciencia_early_stopping": 25,
    "paciencia_reduce_lr": 10,
    "factor_reduce_lr": 0.5,
    "semillas_modelo": [0, 1, 2, 3, 4],   # repeticiones para media y desviación
}

DATA_DIR = os.environ.get("MODELNET_DIR", "ModelNet10")
OUT_DIR = "salidas"
os.makedirs(OUT_DIR, exist_ok=True)

CLASES = ["bathtub", "bed", "chair", "desk", "dresser",
          "monitor", "night_stand", "sofa", "table", "toilet"]
CLASES_ES = ["bañera", "cama", "silla", "escritorio", "cómoda",
             "monitor", "mesa de noche", "sofá", "mesa", "sanitario"]
MOBILIARIO = ["bed", "chair", "desk", "dresser", "night_stand", "sofa", "table"]

# Estilo de figuras: tinta oscura, rejilla tenue, dos colores para series
AZUL, NARANJA = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300, "font.size": 9,
    "axes.edgecolor": "#52514e", "axes.labelcolor": "#0b0b0b",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": "#e4e3df", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
})

## 2. Descarga de ModelNet10
Se usa la copia oficial del grupo de Princeton (Wu et al., 2015). Si el enlace falla, se puede
descargar `ModelNet10.zip` a mano desde https://modelnet.cs.princeton.edu, ponerlo en la carpeta del
proyecto y volver a ejecutar la celda.

In [ ]:
URLS = [
    "http://3dvision.princeton.edu/projects/2014/3DShapeNets/ModelNet10.zip",
    "https://3dvision.princeton.edu/projects/2014/3DShapeNets/ModelNet10.zip",
]

if not os.path.isdir(DATA_DIR):
    if not os.path.exists("ModelNet10.zip"):
        for url in URLS:
            try:
                print("Descargando", url)
                urllib.request.urlretrieve(url, "ModelNet10.zip")
                break
            except Exception as e:
                print("  falló:", e)
    with zipfile.ZipFile("ModelNet10.zip") as z:
        z.extractall(".")
print("Carpeta de datos:", os.path.abspath(DATA_DIR))

## 3. Lectura de mallas OFF y muestreo de la superficie
Algunos archivos de ModelNet traen la cabecera pegada a los conteos (`OFF1234 567 0`),
por eso se usa un lector propio. El muestreo es uniforme por área: cada triángulo
recibe puntos en proporción a su superficie.

In [ ]:
def leer_off(ruta):
    with open(ruta, "r") as f:
        texto = f.read()
    tokens = texto.split()
    if tokens[0] == "OFF":
        i = 1
    elif tokens[0].startswith("OFF"):      # cabecera pegada
        tokens[0] = tokens[0][3:]
        i = 0
    else:
        raise ValueError("Archivo OFF no válido: " + ruta)
    nv, nf = int(tokens[i]), int(tokens[i + 1])
    i += 3
    V = np.array(tokens[i:i + 3 * nv], dtype=np.float64).reshape(nv, 3)
    i += 3 * nv
    caras = []
    resto = tokens[i:]
    j = 0
    for _ in range(nf):
        n = int(resto[j]); idx = [int(t) for t in resto[j + 1:j + 1 + n]]
        j += 1 + n
        for k in range(1, n - 1):          # triangulación en abanico
            caras.append((idx[0], idx[k], idx[k + 1]))
    return V, np.array(caras, dtype=np.int64)


def muestrear_superficie(V, F, n, rng):
    tri = V[F]
    areas = 0.5 * np.linalg.norm(np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0]), axis=1)
    if areas.sum() <= 0:
        return V[rng.choice(len(V), n, replace=True)]
    elegidos = rng.choice(len(F), n, p=areas / areas.sum())
    r1, r2 = rng.random(n), rng.random(n)
    s = np.sqrt(r1)
    a, b, c = tri[elegidos, 0], tri[elegidos, 1], tri[elegidos, 2]
    return (1 - s)[:, None] * a + (s * (1 - r2))[:, None] * b + (s * r2)[:, None] * c


def normalizar(P):
    """Centra en el centroide y escala a la esfera unitaria."""
    P = P - P.mean(axis=0)
    return P / (np.linalg.norm(P, axis=1).max() + 1e-12)

## 4. Vector de características (146 dimensiones)

| Grupo | Contenido | Dim. |
|---|---|---|
| Forma global | extensión de la caja englobante, razones de aspecto, desviación, asimetría y curtosis por eje, rasgos de autovalores de la covarianza | 24 |
| Distribuciones | histogramas de ocupación por eje, D1 (distancia al centroide), D2 (distancia entre pares) | 96 |
| Locales | histogramas de linealidad, planaridad y esfericidad en vecindarios de k puntos, media y desviación del cambio de curvatura | 26 |

In [ ]:
EPS = 1e-12

def rasgos_autovalores(ev):
    """ev ordenados de mayor a menor (..., 3). Devuelve rasgos tipo Weinmann et al."""
    ev = np.clip(ev, EPS, None)
    e = ev / ev.sum(axis=-1, keepdims=True)
    e1, e2, e3 = e[..., 0], e[..., 1], e[..., 2]
    lin = (e1 - e2) / e1
    pla = (e2 - e3) / e1
    esf = e3 / e1
    omni = np.cbrt(e1 * e2 * e3)
    anis = (e1 - e3) / e1
    entr = -(e * np.log(e)).sum(axis=-1)
    return e1, e2, e3, lin, pla, esf, omni, anis, entr


def histograma(x, rango, bins):
    h, _ = np.histogram(x, bins=bins, range=rango)
    return h / max(h.sum(), 1)


def extraer_caracteristicas(P, rng, cfg=CONFIG):
    # --- forma global (24) ---
    ext = P.max(0) - P.min(0)
    razones = np.array([ext[0] / (ext[1] + EPS), ext[2] / (ext[0] + EPS), ext[2] / (ext[1] + EPS)])
    momentos = np.concatenate([P.std(0), stats.skew(P, axis=0), stats.kurtosis(P, axis=0)])
    ev = np.sort(np.linalg.eigvalsh(np.cov(P.T)))[::-1]
    autov = np.array(rasgos_autovalores(ev))
    f_global = np.concatenate([ext, razones, momentos, autov])

    # --- distribuciones (96) ---
    h_ejes = np.concatenate([histograma(P[:, d], (-1, 1), cfg["bins_eje"]) for d in range(3)])
    d1 = np.linalg.norm(P, axis=1)
    h_d1 = histograma(d1, (0, 1), cfg["bins_d1"])
    i, j = rng.integers(0, len(P), (2, cfg["n_pares_d2"]))
    d2 = np.linalg.norm(P[i] - P[j], axis=1)
    h_d2 = histograma(d2, (0, 2), cfg["bins_d2"])
    f_dist = np.concatenate([h_ejes, h_d1, h_d2])

    # --- locales (26) ---
    _, vec = cKDTree(P).query(P, k=cfg["k_vecinos"])
    Q = P[vec] - P[vec].mean(axis=1, keepdims=True)
    C = np.einsum("nki,nkj->nij", Q, Q) / cfg["k_vecinos"]
    evl = np.linalg.eigvalsh(C)[:, ::-1]
    e1, e2, e3, lin, pla, esf, *_ = rasgos_autovalores(evl)
    curv = e3  # cambio de curvatura = e3 normalizado
    b = cfg["bins_local"]
    f_local = np.concatenate([histograma(lin, (0, 1), b), histograma(pla, (0, 1), b),
                              histograma(esf, (0, 1), b), [curv.mean(), curv.std()]])
    return np.concatenate([f_global, f_dist, f_local]).astype(np.float32)


GRUPOS = {"global": slice(0, 24), "distribuciones": slice(24, 120), "locales": slice(120, 146)}

## 5. Procesamiento de todo el conjunto
Se respeta la partición oficial (`train`/`test`). Las nubes se guardan en caché (`.npz`)
para no repetir el muestreo si se vuelve a ejecutar el notebook.

In [ ]:
CACHE = os.path.join(OUT_DIR, "cache_modelnet10.npz")

def procesar(cfg=CONFIG):
    rng = np.random.default_rng(cfg["semilla_datos"])
    X, y, split, nubes, nombres = [], [], [], [], []
    t0 = time.time()
    for c, clase in enumerate(CLASES):
        for parte in ["train", "test"]:
            carpeta = os.path.join(DATA_DIR, clase, parte)
            for nombre in sorted(f for f in os.listdir(carpeta) if f.endswith(".off")):
                V, F = leer_off(os.path.join(carpeta, nombre))
                P = normalizar(muestrear_superficie(V, F, cfg["n_puntos"], rng))
                X.append(extraer_caracteristicas(P, rng)); y.append(c)
                split.append(parte); nubes.append(P.astype(np.float16)); nombres.append(nombre)
        print(f"{clase:12s} listo ({time.time() - t0:.0f} s)")
    return (np.stack(X), np.array(y), np.array(split), np.stack(nubes), np.array(nombres),
            time.time() - t0)

if os.path.exists(CACHE):
    d = np.load(CACHE)
    X, y, split, nubes, nombres = d["X"], d["y"], d["split"], d["nubes"], d["nombres"]
    t_extraccion = float(d["t"])
else:
    X, y, split, nubes, nombres, t_extraccion = procesar()
    np.savez_compressed(CACHE, X=X, y=y, split=split, nubes=nubes, nombres=nombres, t=t_extraccion)

print("Matriz de características:", X.shape, "| tiempo de extracción:", round(t_extraccion), "s")
assert np.isfinite(X).all(), "Hay valores no finitos en las características"

## 6. Particiones y estandarización (z-score)
La validación sale del entrenamiento oficial de forma estratificada. La media y la desviación
se calculan **solo** con el subconjunto de entrenamiento, para no filtrar información de validación ni de prueba.

In [ ]:
idx_tr_full = np.where(split == "train")[0]
idx_te = np.where(split == "test")[0]
idx_tr, idx_va = train_test_split(idx_tr_full, test_size=CONFIG["frac_validacion"],
                                  stratify=y[idx_tr_full], random_state=CONFIG["semilla_datos"])

mu = X[idx_tr].mean(0)
sigma = X[idx_tr].std(0) + 1e-8
Z = (X - mu) / sigma

conteos = {CLASES[c]: {"train": int(((y == c) & (split == "train")).sum()),
                       "test": int(((y == c) & (split == "test")).sum())} for c in range(10)}
print("entrenamiento:", len(idx_tr), "validación:", len(idx_va), "prueba:", len(idx_te))

pesos_clase = compute_class_weight("balanced", classes=np.arange(10), y=y[idx_tr])
pesos_clase = {i: float(w) for i, w in enumerate(pesos_clase)}

## 7. Modelos
* **Línea base lineal:** una sola capa densa con softmax (equivale a regresión logística multiclase).
* **MLP propuesto:** tres capas ocultas Dense → BatchNorm → ReLU → Dropout, regularización L2 y salida softmax.

In [ ]:
def construir_mlp(dim_entrada, cfg=CONFIG, lineal=False):
    reg = regularizers.l2(cfg["l2"])
    ent = keras.Input(shape=(dim_entrada,))
    h = ent
    if not lineal:
        for n, p in zip(cfg["capas_ocultas"], cfg["dropout"]):
            h = layers.Dense(n, kernel_initializer="he_normal", kernel_regularizer=reg)(h)
            h = layers.BatchNormalization()(h)
            h = layers.Activation("relu")(h)
            h = layers.Dropout(p)(h)
    sal = layers.Dense(10, activation="softmax", kernel_regularizer=reg)(h)
    modelo = keras.Model(ent, sal)
    modelo.compile(optimizer=keras.optimizers.Adam(cfg["tasa_aprendizaje"]),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo


def entrenar(columnas, semilla, lineal=False, cfg=CONFIG, verbose=0):
    keras.utils.set_random_seed(semilla)
    m = construir_mlp(Z[:, columnas].shape[1], cfg, lineal)
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=cfg["paciencia_early_stopping"],
                                        restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=cfg["factor_reduce_lr"],
                                            patience=cfg["paciencia_reduce_lr"], min_lr=1e-5)]
    t0 = time.time()
    hist = m.fit(Z[idx_tr][:, columnas], y[idx_tr],
                 validation_data=(Z[idx_va][:, columnas], y[idx_va]),
                 epochs=cfg["max_epocas"], batch_size=cfg["batch_size"],
                 class_weight=pesos_clase, callbacks=cb, verbose=verbose)
    t_ent = time.time() - t0
    t0 = time.time()
    prob = m.predict(Z[idx_te][:, columnas], verbose=0)
    t_inf = (time.time() - t0) / len(idx_te)
    return m, hist.history, prob, t_ent, t_inf


def metricas(prob, y_true):
    y_pred = prob.argmax(1)
    return {"OA": accuracy_score(y_true, y_pred),
            "mAcc": balanced_accuracy_score(y_true, y_pred),
            "F1_macro": f1_score(y_true, y_pred, average="macro")}

## 8. Experimentos
Cada configuración se entrena con 5 semillas. Se reporta media ± desviación estándar.

In [ ]:
TODAS = np.arange(146)
EXPERIMENTOS = {
    "lineal_todas":           (TODAS, True),
    "mlp_global":             (np.arange(146)[GRUPOS["global"]], False),
    "mlp_global_dist":        (np.arange(146)[:120], False),
    "mlp_todas (propuesto)":  (TODAS, False),
}

resultados, guardado, modelos = {}, {}, {}
for nombre, (cols, lineal) in EXPERIMENTOS.items():
    filas, modelos[nombre] = [], []
    for s in CONFIG["semillas_modelo"]:
        m, h, prob, t_ent, t_inf = entrenar(cols, s, lineal)
        r = metricas(prob, y[idx_te]); r.update(epocas=len(h["loss"]), t_ent=t_ent, t_inf_ms=1000 * t_inf)
        filas.append(r)
        modelos[nombre].append(m)          # se guardan para la prueba de robustez
        if s == CONFIG["semillas_modelo"][0]:
            guardado[nombre] = (m, h, prob)
    resumen = {k: (float(np.mean([f[k] for f in filas])), float(np.std([f[k] for f in filas])))
               for k in filas[0]}
    resumen["parametros"] = int(guardado[nombre][0].count_params())
    resultados[nombre] = {"resumen": resumen, "por_semilla": filas}
    print(f"{nombre:24s} OA={resumen['OA'][0]*100:.2f}±{resumen['OA'][1]*100:.2f}  "
          f"mAcc={resumen['mAcc'][0]*100:.2f}  F1={resumen['F1_macro'][0]*100:.2f}  "
          f"épocas={resumen['epocas'][0]:.0f}  params={resumen['parametros']}")

## 9. Análisis detallado del modelo propuesto (semilla 0)

In [ ]:
modelo, hist, prob = guardado["mlp_todas (propuesto)"]
y_te = y[idx_te]
y_pred = prob.argmax(1)
modelo.summary()

p, r, f, sop = precision_recall_fscore_support(y_te, y_pred, labels=range(10), zero_division=0)
por_clase = {CLASES[c]: {"precision": float(p[c]), "recall": float(r[c]), "f1": float(f[c]),
                         "soporte": int(sop[c])} for c in range(10)}
cm = confusion_matrix(y_te, y_pred, labels=range(10))

pares = [(CLASES[i], CLASES[j], int(cm[i, j])) for i in range(10) for j in range(10) if i != j and cm[i, j] > 0]
pares.sort(key=lambda t: -t[2])
print("Confusiones más frecuentes:", pares[:6])

mask_mob = np.isin(y_te, [CLASES.index(c) for c in MOBILIARIO])
oa_mobiliario = float((y_pred[mask_mob] == y_te[mask_mob]).mean())

# --- Verificación: ¿la pieza corresponde a la categoría esperada? ---
# Pares conformes: (pieza, su clase real). Pares no conformes: (pieza, cada una de las otras 9 clases).
puntaje_pos = prob[np.arange(len(y_te)), y_te]
mascara_neg = np.ones_like(prob, dtype=bool); mascara_neg[np.arange(len(y_te)), y_te] = False
puntaje_neg = prob[mascara_neg]
etiquetas = np.r_[np.ones_like(puntaje_pos), np.zeros_like(puntaje_neg)]
puntajes = np.r_[puntaje_pos, puntaje_neg]
auc = roc_auc_score(etiquetas, puntajes)
fpr, tpr, umbrales = roc_curve(etiquetas, puntajes)
k = np.argmin(np.abs(fpr - (1 - tpr)))
eer = float((fpr[k] + 1 - tpr[k]) / 2)
# Regla simple de la estación: aceptar si la clase predicha coincide con la orden
frr_argmax = float((y_pred != y_te).mean())
far_argmax = float((prob.argmax(1)[:, None] == np.arange(10)[None, :])[mascara_neg.reshape(prob.shape)].mean())
print(f"AUC verificación={auc:.4f}  EER={eer*100:.2f}%  FRR(argmax)={frr_argmax*100:.2f}%  FAR(argmax)={far_argmax*100:.3f}%")

## 10. Figuras

In [ ]:
# 10.1 Distribución de clases
fig, ax = plt.subplots(figsize=(6.5, 2.6))
xpos = np.arange(10)
tr = [conteos[c]["train"] for c in CLASES]; te = [conteos[c]["test"] for c in CLASES]
ax.bar(xpos - 0.2, tr, 0.38, color=AZUL, label="Entrenamiento")
ax.bar(xpos + 0.2, te, 0.38, color=NARANJA, label="Prueba")
ax.set_xticks(xpos, CLASES_ES, rotation=30, ha="right"); ax.set_ylabel("Número de modelos")
ax.legend(frameon=False); ax.grid(axis="x", visible=False)
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_distribucion_clases.pdf"); plt.show()

In [ ]:
# 10.2 Un ejemplo de nube por clase (color = altura, un solo tono)
def dibujar_nube(ax, P, titulo):
    P = P.astype(np.float32)
    ax.scatter(P[:, 0], P[:, 1], P[:, 2], c=P[:, 2], cmap="Blues", vmin=-1.3, vmax=1, s=1.2, linewidths=0)
    ax.set_title(titulo, fontsize=8); ax.set_axis_off()
    ax.set_xlim(-0.8, 0.8); ax.set_ylim(-0.8, 0.8); ax.set_zlim(-0.8, 0.8)
    ax.view_init(elev=20, azim=-60)

fig = plt.figure(figsize=(7, 3.2))
for c in range(10):
    i = np.where((y == c) & (split == "train"))[0][0]
    dibujar_nube(fig.add_subplot(2, 5, c + 1, projection="3d"), nubes[i], CLASES_ES[c])
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_ejemplos_nubes.pdf"); plt.show()

In [ ]:
# 10.3 Curvas de entrenamiento
fig, axs = plt.subplots(1, 2, figsize=(6.5, 2.4))
ep = np.arange(1, len(hist["loss"]) + 1)
axs[0].plot(ep, hist["loss"], color=AZUL, lw=1.5, label="Entrenamiento")
axs[0].plot(ep, hist["val_loss"], color=NARANJA, lw=1.5, label="Validación")
axs[0].set_xlabel("Época"); axs[0].set_ylabel("Entropía cruzada")
axs[1].plot(ep, hist["accuracy"], color=AZUL, lw=1.5, label="Entrenamiento")
axs[1].plot(ep, hist["val_accuracy"], color=NARANJA, lw=1.5, label="Validación")
axs[1].set_xlabel("Época"); axs[1].set_ylabel("Exactitud")
axs[1].legend(frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_curvas_entrenamiento.pdf"); plt.show()

In [ ]:
# 10.4 Matriz de confusión normalizada por fila
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(4.6, 4.0))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                    color="white" if cmn[i, j] > 0.6 else "#0b0b0b")
ax.set_xticks(range(10), CLASES_ES, rotation=45, ha="right"); ax.set_yticks(range(10), CLASES_ES)
ax.set_xlabel("Clase predicha"); ax.set_ylabel("Clase real"); ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Proporción de la fila")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_matriz_confusion.pdf"); plt.show()

In [ ]:
# 10.5 Ejemplos cualitativos: 5 aciertos y 5 errores del conjunto de prueba
conf = prob.max(1)
rng_fig = np.random.default_rng(7)
aciertos = rng_fig.choice(np.where(y_pred == y_te)[0], 5, replace=False)
errores = np.where(y_pred != y_te)[0]
errores = errores[np.argsort(-conf[errores])][:5]
fig = plt.figure(figsize=(7, 3.4))
for n, i in enumerate(np.r_[aciertos, errores]):
    marca = "correcto" if y_pred[i] == y_te[i] else "error"
    titulo = f"real: {CLASES_ES[y_te[i]]}\npred: {CLASES_ES[y_pred[i]]} ({conf[i]*100:.0f}%)\n[{marca}]"
    dibujar_nube(fig.add_subplot(2, 5, n + 1, projection="3d"), nubes[idx_te[i]], titulo)
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_predicciones.pdf"); plt.show()
ejemplos = [{"archivo": str(nombres[idx_te[i]]), "real": CLASES[y_te[i]], "pred": CLASES[y_pred[i]],
             "confianza": float(conf[i])} for i in np.r_[aciertos, errores]]

In [ ]:
# 10.6 Curva ROC de la verificación
fig, ax = plt.subplots(figsize=(3.2, 3.0))
ax.plot(fpr, tpr, color=AZUL, lw=1.8, label=f"MLP (AUC = {auc:.3f})")
xx = np.logspace(-4, 0, 200)
ax.plot(xx, xx, color="#9a9893", lw=1, ls="--", label="Azar")  # diagonal en escala log
ax.set_xscale("log"); ax.set_xlim(1e-4, 1)
ax.set_xlabel("Tasa de falsa aceptación (escala log)"); ax.set_ylabel("Tasa de aceptación correcta")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_roc_verificacion.pdf"); plt.show()

## 11. Prueba de robustez

ModelNet10 tiene modelos CAD limpios, completos y alineados. Un escáner en la planta entrega algo distinto:
la pieza puede llegar girada sobre la banda, el sensor agrega ruido y una parte de la pieza queda tapada porque
el escáner la ve desde un solo lado. Para medir qué tanto afecta cada cosa, se toman las 908 nubes de prueba,
se les aplica una perturbación, se vuelven a calcular las 146 características y se evalúan **los mismos
modelos ya entrenados** (los 5 de cada configuración). Los modelos no se vuelven a entrenar.

| Perturbación | Qué simula | Niveles |
|---|---|---|
| Rotación alrededor del eje vertical | La pieza llega girada sobre la banda | 0°, 30°, 90° y 180° |
| Ruido gaussiano en cada punto ($\sigma$, en unidades del radio del objeto) | Error de medición del sensor | 0, 0,01, 0,02 y 0,04 |
| Parte tapada | El escáner solo ve la pieza desde un lado | 0 %, 10 %, 25 % y 40 % de los puntos |

Para tapar una parte se escoge una dirección horizontal al azar y se quitan los puntos que quedan más
adelante en esa dirección, como si esa cara no se viera. Después de cada perturbación la nube se vuelve a
centrar y escalar, igual que en el preprocesamiento. El nivel 0 es el conjunto de prueba original.

In [ ]:
def rotar_z(P, grados):
    """Gira la nube alrededor del eje vertical (z)."""
    t = np.deg2rad(grados)
    R = np.array([[np.cos(t), -np.sin(t), 0],
                  [np.sin(t),  np.cos(t), 0],
                  [0,          0,         1]])
    return P @ R.T


def agregar_ruido(P, sigma, rng):
    """Suma ruido gaussiano a cada coordenada."""
    return P + rng.normal(0, sigma, P.shape)


def tapar(P, fraccion, rng):
    """Quita la fracción de puntos que queda más adelante en una dirección horizontal al azar."""
    angulo = rng.uniform(0, 2 * np.pi)
    direccion = np.array([np.cos(angulo), np.sin(angulo), 0])
    proyeccion = P @ direccion
    corte = np.quantile(proyeccion, 1 - fraccion)
    return P[proyeccion < corte]


PERTURBACIONES = {"rotación": [0, 30, 90, 180],
                  "ruido": [0, 0.01, 0.02, 0.04],
                  "parte tapada": [0, 0.10, 0.25, 0.40]}


def caracteristicas_perturbadas(tipo, nivel, semilla=123):
    """Aplica la perturbación a las nubes de prueba y devuelve las características estandarizadas."""
    if nivel == 0:
        return Z[idx_te]                      # conjunto de prueba original, sin cambios
    rng = np.random.default_rng(semilla)
    filas = []
    for i in idx_te:
        P = nubes[i].astype(np.float64)
        if tipo == "rotación":
            P = rotar_z(P, nivel)
        elif tipo == "ruido":
            P = agregar_ruido(P, nivel, rng)
        else:
            P = tapar(P, nivel, rng)
        P = normalizar(P)
        filas.append(extraer_caracteristicas(P, rng))
    return (np.stack(filas) - mu) / sigma


# Ejemplo visual: la misma silla con cada perturbación en su nivel más alto
i_ej = idx_te[np.where(y_te == CLASES.index("chair"))[0][0]]
P0 = nubes[i_ej].astype(np.float64)
rng_ej = np.random.default_rng(0)
ejemplos_pert = [("original", P0), ("girada 90°", rotar_z(P0, 90)),
                 ("ruido σ = 0,04", agregar_ruido(P0, 0.04, rng_ej)), ("40 % tapada", tapar(P0, 0.40, rng_ej))]
fig = plt.figure(figsize=(7, 2.0))
for n, (titulo, P) in enumerate(ejemplos_pert):
    dibujar_nube(fig.add_subplot(1, 4, n + 1, projection="3d"), normalizar(P), titulo)
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_perturbaciones.pdf"); plt.show()

Se evalúan las cuatro configuraciones con cada perturbación. Para el modelo propuesto también se calculan las
métricas de verificación (AUC y tasas de error con la regla de aceptar solo si la clase más probable es la
de la orden). Esta celda vuelve a calcular las características de 908 nubes por cada nivel, así que tarda
varios minutos.

In [ ]:
def verificacion(prob, y_true):
    """AUC de verificación y tasas de error con la regla del argmax (igual que en la sección 9)."""
    n = len(y_true)
    pos = prob[np.arange(n), y_true]
    neg_mask = np.ones_like(prob, dtype=bool); neg_mask[np.arange(n), y_true] = False
    auc_v = roc_auc_score(np.r_[np.ones(n), np.zeros(neg_mask.sum())], np.r_[pos, prob[neg_mask]])
    pred = prob.argmax(1)
    frr = (pred != y_true).mean()
    far = (pred[:, None] == np.arange(10)[None, :])[neg_mask].mean()
    return auc_v, frr, far


robustez, filas_tabla = {}, []
for tipo, niveles in PERTURBACIONES.items():
    robustez[tipo] = {}
    for nivel in niveles:
        t0 = time.time()
        Zp = caracteristicas_perturbadas(tipo, nivel)
        r = {}
        for nombre, (cols, _) in EXPERIMENTOS.items():
            probs = [m.predict(Zp[:, cols], verbose=0) for m in modelos[nombre]]
            oa = [accuracy_score(y_te, p.argmax(1)) for p in probs]
            r[nombre] = {"OA": (float(np.mean(oa)), float(np.std(oa)))}
            if nombre == "mlp_todas (propuesto)":
                v = np.array([verificacion(p, y_te) for p in probs])
                r[nombre].update(AUC=float(v[:, 0].mean()), FRR=float(v[:, 1].mean()), FAR=float(v[:, 2].mean()),
                                 mAcc=float(np.mean([balanced_accuracy_score(y_te, p.argmax(1)) for p in probs])))
        robustez[tipo][str(nivel)] = r
        prop = r["mlp_todas (propuesto)"]
        filas_tabla.append({"perturbación": tipo, "nivel": nivel,
                            **{k: 100 * v["OA"][0] for k, v in r.items()},
                            "AUC (propuesto)": prop["AUC"], "FAR % (propuesto)": 100 * prop["FAR"]})
        print(f"{tipo:13s} nivel {nivel:<5}  OA propuesto = {100 * prop['OA'][0]:.1f} %  ({time.time() - t0:.0f} s)")

tabla_robustez = pd.DataFrame(filas_tabla).round(2)
tabla_robustez

In [ ]:
# Exactitud de cada configuración según el nivel de cada perturbación (media ± desviación de 5 semillas)
COLORES = {"lineal_todas": "#9a9893", "mlp_global": "#2e9e5b",
           "mlp_global_dist": NARANJA, "mlp_todas (propuesto)": AZUL}
ETIQUETAS = {"lineal_todas": "Lineal (146)", "mlp_global": "MLP, forma global (24)",
             "mlp_global_dist": "MLP, global + distrib. (120)", "mlp_todas (propuesto)": "MLP propuesto (146)"}
EJE_X = {"rotación": "Rotación (grados)", "ruido": "Ruido σ", "parte tapada": "Fracción tapada"}
fig, axs = plt.subplots(1, 3, figsize=(7, 2.8), sharey=True)
for ax, (tipo, niveles) in zip(axs, PERTURBACIONES.items()):
    for nombre in EXPERIMENTOS:
        media = [100 * robustez[tipo][str(n)][nombre]["OA"][0] for n in niveles]
        desv = [100 * robustez[tipo][str(n)][nombre]["OA"][1] for n in niveles]
        ax.errorbar(range(len(niveles)), media, yerr=desv, color=COLORES[nombre], lw=1.4,
                    marker="o", ms=3, capsize=2, label=ETIQUETAS[nombre])
    ax.set_xticks(range(len(niveles)), [str(n) for n in niveles])
    ax.set_xlabel(EJE_X[tipo])
axs[0].set_ylabel("Exactitud (%)")
fig.legend(*axs[0].get_legend_handles_labels(), loc="lower center", ncol=4, frameon=False, fontsize=7)
fig.tight_layout(rect=(0, 0.1, 1, 1)); fig.savefig(f"{OUT_DIR}/fig_robustez.pdf"); plt.show()

## 12. Guardar resultados

In [ ]:
salida = {
    "config": CONFIG,
    "dataset": {"conteos": conteos, "n_train": int(len(idx_tr)), "n_val": int(len(idx_va)),
                "n_test": int(len(idx_te)), "dim_caracteristicas": int(X.shape[1]),
                "t_extraccion_s": float(t_extraccion),
                "t_extraccion_ms_por_objeto": float(1000 * t_extraccion / len(X))},
    "experimentos": {k: v["resumen"] for k, v in resultados.items()},
    "por_semilla": {k: v["por_semilla"] for k, v in resultados.items()},
    "propuesto_semilla0": {
        **{k: float(v) for k, v in metricas(prob, y_te).items()},
        "epocas": len(hist["loss"]), "por_clase": por_clase, "matriz_confusion": cm.tolist(),
        "confusiones_top": pares[:8], "OA_mobiliario": oa_mobiliario,
        "verificacion": {"AUC": float(auc), "EER": eer, "FRR_argmax": frr_argmax, "FAR_argmax": far_argmax},
        "ejemplos_cualitativos": ejemplos,
    },
    "robustez": robustez,
    "versiones": {"tensorflow": tf.__version__, "numpy": np.__version__},
}
with open(f"{OUT_DIR}/resultados.json", "w", encoding="utf-8") as fjson:
    json.dump(salida, fjson, indent=2, ensure_ascii=False, default=float)
print("Listo: resultados guardados en", os.path.abspath(OUT_DIR))